In [ ]:
import csv

import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))

csv_path = data_root / "final_audit" / "valid_instances.csv"
with csv_path.open("r", encoding="utf-8-sig", newline="") as stream:
    valid_instances = list(csv.DictReader(stream))

print("Number of instances:", len(valid_instances))
print("Fields:", valid_instances[0].keys())
print("First instance:", valid_instances[0])

In [ ]:
from src.dataset import OCLDataset

In [ ]:
from src.model import load_model

model, preprocess, _ = load_model("ViT-B/32", "cpu", os.environ.get("OCL_WEIGHTS_DIR"))

In [ ]:
dataset = OCLDataset(
    data_root=data_root,
    resource_root=resource_root,
    preprocess=preprocess
)

print(len(dataset))

In [ ]:
sample = dataset[0]

for key, value in sample.items():
    print(key, value.shape, value.dtype)

In [ ]:
import torch

print(dataset.categories[sample["category"].item()])

print(sample["attribute"].sum())
print(sample["affordance"].sum())

print(torch.where(sample["attribute"] == 1)[0])

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    drop_last=False
)

batch = next(iter(test_loader))

for key, value in batch.items():
    print(key, value.shape)

In [ ]:
print(len(dataset))
print(len(test_loader))

last_batch = next(iter(DataLoader(
    dataset,
    batch_size=4,
    sampler=range(len(dataset) - 3, len(dataset))
)))

for key, value in last_batch.items():
    print(key, value.shape)